# Distill the listen decision

Fine-tune a small model to make Polyphony's *listen* decision the way the teacher does, then
export it to Ollama as `polyphony-listen`.

1. Export data locally: `python scripts/export_listen_samples.py --cutoff ... --out data/listen`
2. Upload `train.jsonl` (and `meta.json`) here. **Never upload `test.jsonl` for training** — it is
   the time-held-out real data the student is judged on.
3. Run all cells on a free GPU (Runtime → Change runtime type → T4).
4. Download `polyphony-listen.gguf` and `Modelfile`, then on your machine:
   `ollama create polyphony-listen -f Modelfile` and set `LISTEN_MODEL=polyphony-listen`.
5. Judge it: `python scripts/eval_listen_student.py --model ollama:polyphony-listen ...`

In [ ]:
%%capture
!pip install unsloth

In [ ]:
# Settings
BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct"   # or unsloth/Qwen2.5-0.5B-Instruct for 8 GB machines
MAX_SEQ_LEN = 8192          # listen prompts are a few thousand tokens
LORA_R = 16
EPOCHS = 2
LR = 2e-4
QUANT = "q4_k_m"

In [ ]:
from google.colab import files
uploaded = files.upload()   # train.jsonl, meta.json
import json
meta = json.load(open("meta.json")) if "meta.json" in uploaded else {}
print("cutoff:", meta.get("cutoff"), "| train:", meta.get("train"))

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=0)

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files="train.jsonl", split="train")
ds = ds.map(lambda ex: {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False)})
split = ds.train_test_split(test_size=0.05, seed=0)   # small dev split from TRAIN only
print(split)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=split["train"], eval_dataset=split["test"],
    args=SFTConfig(dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
                   per_device_train_batch_size=1, gradient_accumulation_steps=8,
                   num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type="cosine",
                   warmup_ratio=0.03, logging_steps=10, eval_strategy="epoch",
                   output_dir="out", report_to="none", seed=0))
# Learn only the JSON decision, not the prompt.
trainer = train_on_responses_only(trainer,
    instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
trainer.train()

In [ ]:
# Quick sanity check on a dev example: does it emit valid JSON?
FastLanguageModel.for_inference(model)
ex = split["test"][0]["messages"]
inputs = tokenizer.apply_chat_template(ex[:2], add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=120, temperature=0.7)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
print("teacher:", ex[2]["content"])

In [ ]:
model.save_pretrained_gguf("polyphony-listen", tokenizer, quantization_method=QUANT)
import glob, shutil
gguf = sorted(glob.glob("polyphony-listen*/*.gguf") + glob.glob("polyphony-listen*.gguf"))[-1]
shutil.copy(gguf, "polyphony-listen.gguf")
open("Modelfile", "w").write('''FROM ./polyphony-listen.gguf
TEMPLATE """{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{- range .Messages }}<|im_start|>{{ .Role }}
{{ .Content }}<|im_end|>
{{ end }}<|im_start|>assistant
"""
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.7
PARAMETER num_ctx 8192
''')
files.download("polyphony-listen.gguf"); files.download("Modelfile")